# G1: LoRA training + serving feasibility spike (Gemma 4 31B, L4x4)

Stage A baseline (reused from run 1) -> B training spike on the bf16 QAT checkpoint -> C adapter serving on the scorer's VllmServer. Hard 110-minute watchdog. Plan and pass criteria: docs/STRATEGY.md 'POST-TRAINING PLAN'.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
import json, os, re, shutil, subprocess, sys, threading, time, traceback
from pathlib import Path
WORK = Path('/kaggle/working'); G1 = WORK / 'g1'; CODE = G1 / 'code'; CODE.mkdir(parents=True, exist_ok=True)
RES = G1 / 'results.json'; results = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
BUDGET_MIN = 110
def save_results(): RES.write_text(json.dumps(results, indent=1, default=str))
def _watchdog():   # a hung stage must not burn GPU quota: hard stop (kills the kernel; /kaggle/working is kept)
    results['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
CODE_FILES = {"train_g1.py": "\"\"\"G1 training spike: LoRA SFT of Gemma 4 31B (bf16 QAT-unquantized checkpoint) across the GPUs of one machine.\n\nMeasures what decides feasibility: tokens/s, peak memory per GPU, long-sequence headroom; saves a PEFT adapter in the exact key\nformat the scorer expects. The SAME file is tested locally on a tiny random Gemma 4 (G1_TINY=1) and run on Kaggle on the real model.\n\nEnvironment variables (all strings):\n  G1_MODEL   checkpoint dir (real run) or a dir holding tokenizer + chat template (tiny run)\n  G1_DATA    jsonl of {\"messages\": [...]} (OpenAI-style, dict tool arguments)\n  G1_OUT     output dir (metrics json, adapter_full/)\n  G1_TINY    \"1\" = build a tiny random Gemma 4 instead of loading G1_MODEL weights\n  G1_MAXLEN  max tokens per training sequence (default 16384; median trajectory is ~15.5k tokens, the fixed system+task prompt alone ~4.5k; steps down x0.75 on OOM)\n  G1_RENDER  \"flat\" (default; plain-text tool protocol) or \"native\" (Gemma 4 chat-template tool format; NOT prefix-stable, see STRATEGY.md G1 notes)\n  G1_MICRO   number of micro-steps (sequences) to run (default 16); G1_ACC grad-accumulation (default 4)\n  G1_MAXMEM  JSON dict gpu->limit for device_map auto, e.g. {\"0\":\"15GiB\",\"1\":\"19GiB\",...}\n  G1_TRAIN_SECONDS  stop training after this many seconds (default 1500)\n  G1_PROBES  comma list of long sequence lengths to probe fwd+bwd memory at (default 16384,24576,32768)\n\"\"\"\n\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n# Kaggle's image ships torchao 0.10.0, which peft 0.19.1 rejects with an ImportError while attaching LoRA layers (run 3 failure).\n# We do not use torchao, so hide it: importlib.util.find_spec() returns None for a sys.modules entry of None.\nsys.modules[\"torchao\"] = None\nimport torch\nimport torch.nn.functional as F\nimport transformers\nfrom transformers import AutoTokenizer\n\nMODEL = os.environ[\"G1_MODEL\"]\nDATA = os.environ[\"G1_DATA\"]\nOUT = Path(os.environ[\"G1_OUT\"])\nTINY = os.environ.get(\"G1_TINY\") == \"1\"\nMAXLEN = int(os.environ.get(\"G1_MAXLEN\", \"16384\"))\nRENDER = os.environ.get(\"G1_RENDER\", \"flat\")\nMICRO = int(os.environ.get(\"G1_MICRO\", \"16\"))\nACC = int(os.environ.get(\"G1_ACC\", \"4\"))\nMAXMEM = json.loads(os.environ.get(\"G1_MAXMEM\", \"null\"))\nTRAIN_SECONDS = float(os.environ.get(\"G1_TRAIN_SECONDS\", \"1500\"))\nPROBES = [int(x) for x in os.environ.get(\"G1_PROBES\", \"16384,24576,32768\").split(\",\") if x]\nTARGET_RE = r\".*language_model\\.layers\\.\\d+\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\"\n\nOUT.mkdir(parents=True, exist_ok=True)\nres: dict = {\"stage\": \"start\", \"tiny\": TINY, \"maxlen\": MAXLEN, \"torch\": torch.__version__, \"transformers\": transformers.__version__,\n             \"n_gpus\": torch.cuda.device_count()}\n\n\ndef save() -> None:\n    (OUT / \"train_metrics.json\").write_text(json.dumps(res, indent=1), encoding=\"utf-8\")\n\n\ndef log(*a) -> None:\n    print(\"[g1]\", *a, flush=True)\n\n\ndef gpu_peaks() -> list[float]:\n    return [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n\n\n# ---------------------------------------------------------------- data: chat-template rendering with assistant-only labels\ndef flatten_for_template(msgs: list[dict]) -> list[dict]:\n    \"\"\"Fallback if the chat template rejects tool_calls/tool roles: plain-text tool protocol.\"\"\"\n    out = []\n    for m in msgs:\n        if m[\"role\"] == \"assistant\" and m.get(\"tool_calls\"):\n            calls = \"\\n\".join(json.dumps({\"name\": c[\"function\"][\"name\"], \"arguments\": c[\"function\"][\"arguments\"]}) for c in m[\"tool_calls\"])\n            out.append({\"role\": \"assistant\", \"content\": (m.get(\"content\") or \"\") + \"\\n<tool_call>\\n\" + calls + \"\\n</tool_call>\"})\n        elif m[\"role\"] == \"tool\":\n            out.append({\"role\": \"user\", \"content\": \"<tool_response>\\n\" + (m.get(\"content\") or \"\") + \"\\n</tool_response>\"})\n        else:\n            out.append({\"role\": m[\"role\"], \"content\": m.get(\"content\") or \"\"})\n    return out\n\n\ndef render_example(tok, msgs: list[dict]):\n    def tmpl(m, gen=False):\n        return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=gen)\n\n    full = tmpl(msgs)\n    spans = []\n    for i, m in enumerate(msgs):\n        if m[\"role\"] != \"assistant\":\n            continue\n        # NOT add_generation_prompt: the 31B template's generation prompt appends an empty thought channel that is never rendered\n        # in history, so it can never be a prefix of the rendered turn. Labels therefore also cover the `<|turn>model` header (a few tokens).\n        p, q = tmpl(msgs[:i]), tmpl(msgs[: i + 1])\n        if not (q.startswith(p) and full.startswith(q)):\n            return None, \"template not prefix-stable\"\n        spans.append((len(p), len(q)))\n    enc = tok(full, add_special_tokens=False, return_offsets_mapping=True)\n    ids, off = enc[\"input_ids\"], enc[\"offset_mapping\"]\n    labels = [-100] * len(ids)\n    for k, (a, _b) in enumerate(off):\n        if any(s <= a < e for s, e in spans):\n            labels[k] = ids[k]\n    return (ids, labels), \"ok\"\n\n\ndef build_examples(tok) -> list[tuple[list[int], list[int]]]:\n    rows = [json.loads(line) for line in open(DATA, encoding=\"utf-8\")]\n    exs, why = [], {}\n    mode_counts = {\"native\": 0, \"flattened\": 0}\n    for r in rows:\n        msgs = r[\"messages\"]\n        ex, w, mode = None, \"\", None\n        if RENDER == \"native\":\n            try:\n                ex, w = render_example(tok, msgs)\n                mode = \"native\"\n            except Exception as e:  # noqa: BLE001\n                ex, w = None, f\"native render failed: {type(e).__name__}\"\n        if ex is None:\n            try:\n                ex, w = render_example(tok, flatten_for_template(msgs))\n                mode = \"flattened\"\n            except Exception as e:  # noqa: BLE001\n                ex, w = None, f\"flattened render failed: {type(e).__name__}: {str(e)[:80]}\"\n        if ex is not None and sum(1 for x in ex[1][:MAXLEN] if x != -100) < 200:  # must keep >= 200 label tokens inside the window\n            ex, w = None, \"too few label tokens inside MAXLEN\"\n        if ex is None:\n            why[w] = why.get(w, 0) + 1\n            continue\n        mode_counts[mode] += 1\n        ids, labels = ex\n        exs.append((ids[:32768], labels[:32768]))\n    res[\"data\"] = {\"rows\": len(rows), \"examples\": len(exs), \"render_modes\": mode_counts, \"dropped\": why,\n                   \"tokens_median\": sorted(len(i) for i, _ in exs)[len(exs) // 2] if exs else 0,\n                   \"label_fraction\": round(sum(sum(1 for x in l if x != -100) for _, l in exs) / max(1, sum(len(i) for i, _ in exs)), 3)}\n    log(\"data\", res[\"data\"])\n    return exs\n\n\n# ---------------------------------------------------------------- model\ndef load_model():\n    cls = getattr(transformers, \"Gemma4ForConditionalGeneration\", None) or transformers.AutoModelForImageTextToText\n    if TINY:\n        from transformers import Gemma4Config, Gemma4TextConfig\n        t = Gemma4TextConfig(vocab_size=262144, hidden_size=64, intermediate_size=128, num_hidden_layers=6, num_attention_heads=4,\n                             num_key_value_heads=2, head_dim=16, sliding_window=32,\n                             layer_types=[\"sliding_attention\"] * 5 + [\"full_attention\"], final_logit_softcapping=30.0,\n                             hidden_size_per_layer_input=0, num_kv_shared_layers=0, use_double_wide_mlp=False, max_position_embeddings=65536)\n        cfg = Gemma4Config(text_config=t.to_dict(), vision_config=dict(hidden_size=32, intermediate_size=64, num_hidden_layers=1,\n                           num_attention_heads=2, num_key_value_heads=2, head_dim=16, global_head_dim=16), audio_config=None)\n        return cls(cfg).to(torch.bfloat16).to(\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n    kw = dict(dtype=torch.bfloat16, device_map=\"auto\", attn_implementation=\"sdpa\")\n    if MAXMEM:\n        kw[\"max_memory\"] = {int(k): v for k, v in MAXMEM.items()}\n    return cls.from_pretrained(MODEL, **kw)\n\n\ndef main() -> None:\n    tok = AutoTokenizer.from_pretrained(MODEL)\n    exs = build_examples(tok)\n    save()\n    assert exs, \"no usable training examples\"\n\n    t0 = time.perf_counter()\n    model = load_model()\n    res[\"load_seconds\"] = round(time.perf_counter() - t0, 1)\n    res[\"device_map_gpus\"] = sorted({str(v) for v in getattr(model, \"hf_device_map\", {}).values()}) if hasattr(model, \"hf_device_map\") else [\"single\"]\n    log(\"loaded in\", res[\"load_seconds\"], \"s; mem after load (GiB):\", [round(torch.cuda.memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())])\n    res[\"mem_after_load_gib\"] = [round(torch.cuda.memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n    save()\n\n    from peft import LoraConfig, get_peft_model\n    model.config.use_cache = False\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n    model.enable_input_require_grads()\n    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias=\"none\", target_modules=TARGET_RE, task_type=\"CAUSAL_LM\"))\n    n_tr = sum(p.numel() for p in pm.parameters() if p.requires_grad)\n    res[\"trainable_params_m\"] = round(n_tr / 1e6, 1)\n    names = [n for n, _ in pm.named_modules() if n.endswith(\".lora_A.default\")]\n    res[\"lora_modules\"] = len(names)\n    log(\"trainable params (M):\", res[\"trainable_params_m\"], \"| lora modules:\", len(names), \"| e.g.\", names[:2])\n    assert names, \"TARGET_RE matched no module\"\n    pm.train()\n    dev = model.get_input_embeddings().weight.device\n    opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.0)\n    extra_kw: dict = {}\n\n    from torch.utils.checkpoint import checkpoint\n    inner, lm_head = model.model, model.lm_head\n    softcap = getattr(model.config.get_text_config(), \"final_logit_softcapping\", None)\n    res[\"final_logit_softcapping\"] = softcap\n    CH = 512  # positions per lm_head chunk: [512, 262144] fp32 logits = 0.5 GiB, recomputed in backward (checkpoint)\n\n    def chunk_loss(h, tgt):\n        logits = lm_head(h).float()\n        if softcap:\n            logits = torch.tanh(logits / softcap) * softcap\n        return F.cross_entropy(logits, tgt, reduction=\"sum\")\n\n    def fwd_bwd(ids: list[int], labels: list[int]):\n        pos = [t for t in range(len(ids) - 1) if labels[t + 1] != -100]\n        if not pos:\n            return None, 0\n        x = torch.tensor([ids], device=dev)\n        for attempt in range(2):\n            try:\n                hs = inner(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False, **extra_kw).last_hidden_state[0]\n                break\n            except (ValueError, TypeError) as e:\n                if attempt == 0 and \"token_type\" in str(e):\n                    extra_kw[\"token_type_ids\"] = torch.zeros_like(x)\n                    if \"mm_token_type_ids\" in str(e):\n                        extra_kw[\"mm_token_type_ids\"] = torch.zeros_like(x)\n                    log(\"retrying with token_type_ids\")\n                    continue\n                raise\n        hd = lm_head.weight.device\n        h_sel = hs[torch.tensor(pos, device=hs.device)].to(hd)\n        tgt = torch.tensor([ids[t + 1] for t in pos], device=hd)\n        total = 0.0\n        loss = None\n        for a in range(0, len(pos), CH):\n            part = checkpoint(chunk_loss, h_sel[a:a + CH], tgt[a:a + CH], use_reentrant=False)\n            loss = part if loss is None else loss + part\n        loss = loss / len(pos)\n        (loss / ACC).backward()\n        return float(loss.detach()), len(ids)\n\n    for i in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(i)\n    res[\"stage\"] = \"training\"\n    steps, tok_total, t_train0, losses = [], 0, None, []\n    t_wall0 = time.perf_counter()\n    cur_len = MAXLEN\n    for k in range(MICRO):\n        ids, labels = exs[k % len(exs)]\n        ids, labels = ids[:cur_len], labels[:cur_len]\n        torch.cuda.synchronize() if torch.cuda.is_available() else None\n        t1 = time.perf_counter()\n        try:\n            loss, n_tok = fwd_bwd(ids, labels)\n        except torch.cuda.OutOfMemoryError:\n            res.setdefault(\"oom_events\", []).append({\"micro\": k, \"tokens\": len(ids)})\n            cur_len = int(cur_len * 0.75)\n            res[\"final_train_len\"] = cur_len\n            log(\"OOM at micro\", k, \"tokens\", len(ids), \"-> next sequences capped at\", cur_len)\n            opt.zero_grad(set_to_none=True)\n            torch.cuda.empty_cache()\n            save()\n            continue\n        if (k + 1) % ACC == 0:\n            torch.nn.utils.clip_grad_norm_([p for p in pm.parameters() if p.requires_grad], 1.0)\n            opt.step()\n            opt.zero_grad(set_to_none=True)\n        torch.cuda.synchronize() if torch.cuda.is_available() else None\n        dt = time.perf_counter() - t1\n        if k == 0:\n            t_train0 = time.perf_counter()  # exclude warm-up from throughput\n        else:\n            tok_total += n_tok\n        losses.append(loss)\n        steps.append({\"micro\": k, \"tokens\": n_tok, \"seconds\": round(dt, 2), \"loss\": None if loss is None else round(loss, 4)})\n        el = (time.perf_counter() - t_train0) if t_train0 else 0\n        res.update({\"steps\": steps, \"tokens_per_second\": round(tok_total / el, 1) if el > 0 else None, \"peak_gib_per_gpu\": gpu_peaks()})\n        save()\n        log(f\"micro {k} tokens {n_tok} {dt:.1f}s loss {loss} | tok/s so far {res['tokens_per_second']} | peak GiB {gpu_peaks()}\")\n        if time.perf_counter() - t_wall0 > TRAIN_SECONDS:\n            res[\"stopped_by_time_budget\"] = True\n            break\n\n    # long-sequence headroom (fwd+bwd, no optimizer step)\n    base_ids = [t for ids, _ in exs for t in ids]\n    res[\"probes\"] = []\n    for L in PROBES:\n        if L > len(base_ids):\n            reps = (L // len(base_ids)) + 1\n            seq = (base_ids * reps)[:L]\n        else:\n            seq = base_ids[:L]\n        labels = [-100] * (L - min(2000, L // 4)) + seq[L - min(2000, L // 4):]\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.reset_peak_memory_stats(i)\n        t1 = time.perf_counter()\n        try:\n            fwd_bwd(seq, labels)\n            torch.cuda.synchronize() if torch.cuda.is_available() else None\n            res[\"probes\"].append({\"len\": L, \"ok\": True, \"seconds\": round(time.perf_counter() - t1, 1), \"peak_gib_per_gpu\": gpu_peaks()})\n        except torch.cuda.OutOfMemoryError:\n            res[\"probes\"].append({\"len\": L, \"ok\": False, \"error\": \"OOM\", \"peak_gib_per_gpu\": gpu_peaks()})\n            opt.zero_grad(set_to_none=True)\n            torch.cuda.empty_cache()\n            save()\n            break\n        opt.zero_grad(set_to_none=True)\n        save()\n        log(\"probe\", res[\"probes\"][-1])\n\n    res[\"stage\"] = \"saving\"\n    pm.save_pretrained(str(OUT / \"adapter_full\"))\n    res[\"stage\"] = \"done\"\n    save()\n    log(\"done; adapter saved to\", OUT / \"adapter_full\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except BaseException as e:  # noqa: BLE001\n        res[\"stage\"] = \"FAILED\"\n        res[\"error\"] = f\"{type(e).__name__}: {str(e)[:500]}\"\n        res[\"traceback\"] = traceback.format_exc()[-3000:]\n        save()\n        traceback.print_exc()\n        sys.exit(1)\n", "make_adapters.py": "\"\"\"Turn the raw PEFT output of train_g1.py into scorer-format adapter directories and verify them against the sample adapter format.\n\nVariants written under <out>/:\n  g1_full/    all trained LoRA matrices (attention + MLP)\n  g1_attn/    attention-only subset (q,k,v,o): the safest set for vLLM LoRA support\n  g1_canary/  attention-only with RANDOM lora_B (so the adapter visibly changes outputs even after only a few training steps):\n              used only to prove the serving path really applies adapters (\"silently zero\" LoRA was reported by other teams)\n\nEvery variant: bf16 tensors, keys `base_model.model.model.language_model.layers.N.<module>.lora_{A,B}.weight`, adapter_config.json with\nlist-valued target_modules and base_model_name_or_path = the scorer's model id.\n\nUsage: python make_adapters.py <adapter_full_dir> <out_dir> [sample_adapter_dir]\n\"\"\"\n\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport torch\nfrom safetensors.torch import load_file, save_file\n\nSCORER_MODEL = \"google/gemma-4-31b-it-qat-w4a16-ct\"\nKEY_RE = re.compile(r\"^base_model\\.model\\.model\\.language_model\\.layers\\.(\\d+)\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\\.lora_(A|B)\\.weight$\")\nATTN = (\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\")\n\n\ndef write_variant(out: Path, tensors: dict[str, torch.Tensor], cfg: dict) -> dict:\n    out.mkdir(parents=True, exist_ok=True)\n    mods = sorted({k.split(\".\")[-3] for k in tensors})\n    cfg = dict(cfg, base_model_name_or_path=SCORER_MODEL, inference_mode=True, target_modules=mods)\n    cfg.pop(\"layers_to_transform\", None) if cfg.get(\"layers_to_transform\") is None else None\n    (out / \"adapter_config.json\").write_text(json.dumps(cfg, indent=2), encoding=\"utf-8\")\n    save_file({k: v.to(torch.bfloat16).contiguous() for k, v in tensors.items()}, str(out / \"adapter_model.safetensors\"), metadata={\"format\": \"pt\"})\n    size = (out / \"adapter_model.safetensors\").stat().st_size\n    return {\"dir\": str(out), \"tensors\": len(tensors), \"target_modules\": mods, \"bytes\": size}\n\n\ndef main(src: Path, out: Path, sample: Path | None) -> dict:\n    t = load_file(str(src / \"adapter_model.safetensors\"))\n    cfg = json.loads((src / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n    bad = [k for k in t if not KEY_RE.match(k)]\n    assert not bad, f\"unexpected tensor names, e.g. {bad[:3]}\"\n    report: dict = {\"n_tensors\": len(t), \"rank\": cfg.get(\"r\"), \"alpha\": cfg.get(\"lora_alpha\"), \"dtype_in\": str(next(iter(t.values())).dtype)}\n    if sample is not None and (sample / \"adapter_model.safetensors\").exists():\n        sk = list(load_file(str(sample / \"adapter_model.safetensors\")))\n        report[\"sample_key_example\"] = sk[0]\n        report[\"our_key_example\"] = sorted(t)[0]\n        # same naming scheme (prefix up to the layer index) as the scorer's sample adapter\n        pre = lambda k: k.split(\".layers.\")[0]  # noqa: E731\n        assert pre(sk[0]) == pre(sorted(t)[0]), (pre(sk[0]), pre(sorted(t)[0]))\n        sample_cfg = json.loads((sample / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n        report[\"sample_cfg_keys_missing_in_ours\"] = sorted(set(sample_cfg) - set(cfg))\n    report[\"g1_full\"] = write_variant(out / \"g1_full\", t, cfg)\n    attn = {k: v for k, v in t.items() if k.split(\".\")[-3] in ATTN}\n    report[\"g1_attn\"] = write_variant(out / \"g1_attn\", attn, cfg)\n    g = torch.Generator().manual_seed(0)\n    canary = {k: (torch.randn(v.shape, generator=g) * 0.02 if k.endswith(\"lora_B.weight\") else v.clone()) for k, v in attn.items()}\n    report[\"g1_canary\"] = write_variant(out / \"g1_canary\", canary, cfg)\n    (out / \"adapters_report.json\").write_text(json.dumps(report, indent=1), encoding=\"utf-8\")\n    return report\n\n\nif __name__ == \"__main__\":\n    r = main(Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]) if len(sys.argv) > 3 else None)\n    print(json.dumps(r, indent=1))\n", "g1_probe.py": "\"\"\"HTTP probes against an OpenAI-compatible vLLM server: decode speed, TTFT, concurrency, long-prompt (hang) check, adapter-activity check.\n\nPure `requests` + stdlib so it can be tested against a mock server locally (see test_probe_mock.py) before spending GPU time.\n\"\"\"\n\nimport json\nimport statistics\nimport threading\nimport time\n\nimport requests\n\n\ndef list_models(base_url: str) -> list[str]:\n    r = requests.get(f\"{base_url}/models\", timeout=30)\n    r.raise_for_status()\n    return [m[\"id\"] for m in r.json()[\"data\"]]\n\n\ndef stream_completion(base_url: str, model: str, messages: list[dict], max_tokens: int = 256, min_tokens: int | None = None,\n                      timeout: float = 600, extra: dict | None = None) -> dict:\n    \"\"\"One streamed chat completion; returns timing split into time-to-first-token and steady decode speed.\"\"\"\n    body = {\"model\": model, \"messages\": messages, \"max_tokens\": max_tokens, \"temperature\": 0.0, \"stream\": True,\n            \"stream_options\": {\"include_usage\": True}}\n    if min_tokens:\n        body[\"min_tokens\"] = min_tokens\n    body.update(extra or {})\n    t0 = time.perf_counter()\n    t_first = t_last = None\n    n_chunks, usage = 0, {}\n    with requests.post(f\"{base_url}/chat/completions\", json=body, stream=True, timeout=timeout) as r:\n        if r.status_code != 200:\n            return {\"ok\": False, \"status\": r.status_code, \"error\": r.text[:300]}\n        for raw in r.iter_lines():\n            if not raw or not raw.startswith(b\"data: \"):\n                continue\n            payload = raw[6:]\n            if payload == b\"[DONE]\":\n                break\n            ev = json.loads(payload)\n            if ev.get(\"usage\"):\n                usage = ev[\"usage\"]\n            for ch in ev.get(\"choices\", []):\n                d = ch.get(\"delta\", {})\n                if d.get(\"content\") or d.get(\"reasoning_content\") or d.get(\"reasoning\") or d.get(\"tool_calls\"):\n                    now = time.perf_counter()\n                    t_first = t_first or now\n                    t_last = now\n                    n_chunks += 1\n    t_end = time.perf_counter()\n    comp = usage.get(\"completion_tokens\", n_chunks)\n    out = {\"ok\": True, \"prompt_tokens\": usage.get(\"prompt_tokens\"), \"completion_tokens\": comp, \"total_s\": round(t_end - t0, 2),\n           \"ttft_s\": round(t_first - t0, 2) if t_first else None}\n    if t_first and t_last and t_last > t_first and comp > 1:\n        out[\"decode_tok_s\"] = round((comp - 1) / (t_last - t_first), 2)\n    return out\n\n\ndef decode_probe(base_url: str, model: str, messages: list[dict], reps: int = 3, tokens: int = 256) -> dict:\n    \"\"\"Warm-up + `reps` fixed-length generations at concurrency 1 (the scorer is sequential, so this is the number that matters).\"\"\"\n    stream_completion(base_url, model, messages, max_tokens=32)  # warm-up (also warms LoRA kernels / prefix cache)\n    runs = [stream_completion(base_url, model, messages, max_tokens=tokens, min_tokens=tokens) for _ in range(reps)]\n    good = [r for r in runs if r.get(\"ok\") and r.get(\"decode_tok_s\")]\n    return {\"model\": model, \"runs\": runs,\n            \"decode_tok_s_median\": statistics.median(r[\"decode_tok_s\"] for r in good) if good else None,\n            \"ttft_s_median\": statistics.median(r[\"ttft_s\"] for r in good) if good else None}\n\n\ndef concurrency_probe(base_url: str, model: str, messages: list[dict], n: int = 4, tokens: int = 200) -> dict:\n    results: list[dict] = []\n    lock = threading.Lock()\n\n    def one():\n        r = stream_completion(base_url, model, messages, max_tokens=tokens, min_tokens=tokens)\n        with lock:\n            results.append(r)\n\n    t0 = time.perf_counter()\n    ths = [threading.Thread(target=one) for _ in range(n)]\n    [t.start() for t in ths]\n    [t.join() for t in ths]\n    wall = time.perf_counter() - t0\n    toks = sum(r.get(\"completion_tokens\", 0) for r in results if r.get(\"ok\"))\n    return {\"model\": model, \"n\": n, \"wall_s\": round(wall, 2), \"aggregate_tok_s\": round(toks / wall, 1), \"per_request\": results}\n\n\ndef long_prompt_probe(base_url: str, model: str, target_chars: int, timeout: float = 420) -> dict:\n    \"\"\"Does a long prompt (agent conversations reach 20k+ tokens) complete, or sit in the waiting queue forever?\"\"\"\n    filler = \"\".join(f\"def helper_{i}(x):\\n    return x + {i}  # utility {i}\\n\\n\" for i in range(target_chars // 50))\n    msgs = [{\"role\": \"user\", \"content\": filler + \"\\nSummarize the above in one word.\"}]\n    try:\n        return {\"model\": model, \"target_chars\": target_chars, **stream_completion(base_url, model, msgs, max_tokens=16, timeout=timeout)}\n    except requests.exceptions.RequestException as e:\n        return {\"model\": model, \"ok\": False, \"error\": f\"{type(e).__name__}: {str(e)[:120]}\", \"timeout_s\": timeout}\n\n\nLOGPROB_PROMPTS = [\"Write a Python function that reverses a singly linked list.\",\n                   \"Explain in two sentences what a race condition is.\",\n                   \"Fix the bug: def add(a, b): return a - b\"]\n\n\ndef logprob_probe(base_url: str, model: str, n_tokens: int = 24) -> dict:\n    outs = []\n    for p in LOGPROB_PROMPTS:\n        body = {\"model\": model, \"messages\": [{\"role\": \"user\", \"content\": p}], \"max_tokens\": n_tokens, \"temperature\": 0.0,\n                \"logprobs\": True, \"top_logprobs\": 3}\n        r = requests.post(f\"{base_url}/chat/completions\", json=body, timeout=300)\n        if r.status_code != 200:\n            outs.append({\"error\": r.text[:200]})\n            continue\n        ch = r.json()[\"choices\"][0]\n        lp = ch.get(\"logprobs\", {}).get(\"content\") or []\n        outs.append({\"text\": ch[\"message\"].get(\"content\") or ch[\"message\"].get(\"reasoning_content\") or \"\",\n                     \"tokens\": [x[\"token\"] for x in lp], \"logprobs\": [x[\"logprob\"] for x in lp]})\n    return {\"model\": model, \"outs\": outs}\n\n\ndef compare_logprobs(a: dict, b: dict) -> dict:\n    \"\"\"Is model b different from model a? Compares tokens and per-token logprobs over the common prefix of identical tokens.\"\"\"\n    res = {\"prompts\": []}\n    for oa, ob in zip(a[\"outs\"], b[\"outs\"]):\n        if \"error\" in oa or \"error\" in ob:\n            res[\"prompts\"].append({\"error\": oa.get(\"error\") or ob.get(\"error\")})\n            continue\n        same_text = oa[\"text\"] == ob[\"text\"]\n        k = 0\n        while k < min(len(oa[\"tokens\"]), len(ob[\"tokens\"])) and oa[\"tokens\"][k] == ob[\"tokens\"][k]:\n            k += 1\n        max_d = max((abs(x - y) for x, y in zip(oa[\"logprobs\"][:k], ob[\"logprobs\"][:k])), default=0.0)\n        res[\"prompts\"].append({\"same_text\": same_text, \"identical_prefix_tokens\": k, \"max_abs_logprob_diff_on_prefix\": round(max_d, 5)})\n    res[\"any_text_differs\"] = any(not p.get(\"same_text\", True) for p in res[\"prompts\"])\n    res[\"max_logprob_diff\"] = max((p.get(\"max_abs_logprob_diff_on_prefix\", 0) for p in res[\"prompts\"]), default=0)\n    return res\n", "g1_util.py": "\"\"\"Utilities for the G1 notebook, tested locally (test_g1_util.py) before use on paid GPU time.\n\nstage_checkpoint: copy a checkpoint dir from the slow NFS mount (/kaggle/input: ~90 MB/s per stream, ~320 MB/s with 4 parallel streams)\n                  to local disk with many parallel large-block reads; aborts early on poor throughput.\nrun_supervised:   run a subprocess and kill it if it misses progress milestones (instead of a long fixed timeout that burns GPU quota).\n\"\"\"\n\nimport concurrent.futures as cf\nimport os\nimport shutil\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass StageAbort(RuntimeError):\n    pass\n\n\ndef _copy_block(src: Path, dst: Path, off: int, n: int) -> int:\n    with open(src, \"rb\", buffering=0) as fi, open(dst, \"r+b\", buffering=0) as fo:\n        fi.seek(off)\n        fo.seek(off)\n        left = n\n        while left:\n            buf = fi.read(left)\n            if not buf:\n                raise IOError(f\"short read from {src} at {off + n - left}\")\n            fo.write(buf)\n            left -= len(buf)\n    return n\n\n\ndef stage_checkpoint(src: Path, dst: Path, workers: int = 12, block: int = 64 << 20, min_mbps_after_s: float = 180, min_mbps: float = 30,\n                     max_seconds: float = 1500, log=print) -> dict:\n    dst.mkdir(parents=True, exist_ok=True)\n    jobs, total = [], 0\n    for f in sorted(src.iterdir()):\n        if not f.is_file():\n            continue\n        size = f.stat().st_size\n        total += size\n        if size < block:\n            shutil.copy(f, dst / f.name)\n            continue\n        with open(dst / f.name, \"wb\") as o:\n            o.truncate(size)\n        jobs += [(f, dst / f.name, off, min(block, size - off)) for off in range(0, size, block)]\n    t0, done = time.time(), 0\n    with cf.ThreadPoolExecutor(workers) as ex:\n        futs = [ex.submit(_copy_block, *j) for j in jobs]\n        pending = set(futs)\n        last_log = 0.0\n        while pending:\n            fin, pending = cf.wait(pending, timeout=5, return_when=cf.FIRST_COMPLETED)\n            for fu in fin:\n                done += fu.result()  # re-raises read errors\n            el = time.time() - t0\n            if el - last_log >= 20:\n                last_log = el\n                log(f\"[stage] {done / 1e9:.1f}/{sum(j[3] for j in jobs) / 1e9:.1f} GB, {done / 1e6 / max(el, 1e-9):.0f} MB/s, {el:.0f}s\")\n            if el > min_mbps_after_s and done / 1e6 / el < min_mbps:\n                ex.shutdown(wait=False, cancel_futures=True)\n                raise StageAbort(f\"throughput {done / 1e6 / el:.0f} MB/s < {min_mbps} after {el:.0f}s\")\n            if el > max_seconds:\n                ex.shutdown(wait=False, cancel_futures=True)\n                raise StageAbort(f\"staging exceeded {max_seconds}s\")\n    el = time.time() - t0\n    return {\"bytes\": total, \"seconds\": round(el, 1), \"MBps\": round(sum(j[3] for j in jobs) / 1e6 / max(el, 1e-9), 1), \"workers\": workers}\n\n\ndef run_supervised(cmd: list[str], env: dict, logfile: Path, milestones: list[tuple[str, float]], max_seconds: float, poll: float = 10) -> dict:\n    \"\"\"milestones: [(substring that must appear in the log, deadline in seconds since start)]. Missing a deadline kills the process.\"\"\"\n    t0 = time.time()\n    with open(logfile, \"w\") as lf:\n        p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)\n        reached: dict[str, float] = {}\n        verdict = \"finished\"\n        while p.poll() is None:\n            time.sleep(poll)\n            el = time.time() - t0\n            txt = Path(logfile).read_text(errors=\"ignore\")\n            for pat, deadline in milestones:\n                if pat not in reached and pat in txt:\n                    reached[pat] = round(el, 1)\n            missed = [pat for pat, deadline in milestones if pat not in reached and el > deadline]\n            if missed:\n                verdict = f\"killed: milestone '{missed[0]}' not reached by {dict(milestones)[missed[0]]:.0f}s\"\n                break\n            if el > max_seconds:\n                verdict = f\"killed: exceeded {max_seconds:.0f}s\"\n                break\n        if p.poll() is None:\n            p.kill()\n            p.wait()\n    return {\"returncode\": p.returncode, \"verdict\": verdict, \"seconds\": round(time.time() - t0, 1), \"milestones_reached\": reached}\n"}
for n, txt in CODE_FILES.items(): (CODE / n).write_text(txt, encoding='utf-8')
sys.path.insert(0, str(CODE))
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL_W4 = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
bf16 = sorted(p.parent for p in Path('/kaggle/input').rglob('config.json') if 'unquantized' in str(p).lower())
assert len(bf16) == 1, f'expected one unquantized checkpoint, found {bf16}'
MODEL_BF16 = bf16[0]
G1_DATA = next(Path('/kaggle/input').rglob('g1_train.jsonl'))
print('bf16 checkpoint:', MODEL_BF16, '| w4a16:', MODEL_W4.exists(), '| data:', G1_DATA)
results['gpu'] = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip().splitlines()
print(results['gpu']); save_results()

## Serving helper

In [ ]:
import g1_probe as P
def serve_and_probe(tag, adapter_names):
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
    sub = WORK / f'sub_{tag}'; shutil.rmtree(sub, ignore_errors=True); shutil.copytree(DATA_DIR / 'sample_submission', sub)
    shutil.rmtree(sub / 'adapters', ignore_errors=True)
    manifest = None
    if adapter_names:
        (sub / 'adapters').mkdir()
        for n in adapter_names: shutil.copytree(G1 / 'adapters' / n, sub / 'adapters' / n)
        manifest = discover_adapters(str(sub), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    cfg = VllmConfig(model=str(MODEL_W4), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        max_model_len=32768, dtype='bfloat16', gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True,
        max_loras=8, max_lora_rank=128, tensor_parallel_size=4, startup_timeout=60 * 20)
    srv = VllmServer(cfg, adapter_manifest=manifest)
    rec = {'tag': tag, 'adapters': adapter_names, 'cmd': ' '.join(srv.build_cmd())}
    t0 = time.time()
    try:
        srv.start()
    except Exception as e:
        rec['start_error'] = (str(e) + '\n' + str(getattr(e, 'output', '') or ''))[-3500:]
        return rec
    try:
        rec['boot_s'] = round(time.time() - t0)
        log = Path(srv.log_path).read_text(errors='ignore')
        rec['kv_lines'] = re.findall(r'(?:GPU KV cache size|Available KV cache memory|Maximum concurrency)[^\n]*', log)[:6]
        rec['lora_log_lines'] = [l for l in log.splitlines() if 'lora' in l.lower()][:8]
        ids = P.list_models(srv.base_url); rec['model_ids'] = ids
        base_id = next(i for i in ids if i not in (adapter_names or []))
        first = json.loads(open(G1_DATA, encoding='utf-8').readline())['messages']
        agent_prompt = [m for m in first[:2]]           # real system + task prompt (~4-5k tokens)
        rec['decode'] = {base_id: P.decode_probe(srv.base_url, base_id, agent_prompt)}
        for a in adapter_names or []: rec['decode'][a] = P.decode_probe(srv.base_url, a, agent_prompt)
        rec['concurrency4'] = {base_id: P.concurrency_probe(srv.base_url, base_id, agent_prompt)}
        if adapter_names: rec['concurrency4'][adapter_names[0]] = P.concurrency_probe(srv.base_url, adapter_names[0], agent_prompt)
        rec['long_prompt'] = {base_id: P.long_prompt_probe(srv.base_url, base_id, 40000)}
        if adapter_names: rec['long_prompt'][adapter_names[0]] = P.long_prompt_probe(srv.base_url, adapter_names[0], 40000)
        if adapter_names:
            lp = {i: P.logprob_probe(srv.base_url, i) for i in [base_id] + adapter_names}
            rec['adapter_activity'] = {a: P.compare_logprobs(lp[base_id], lp[a]) for a in adapter_names}
    except Exception:
        rec['probe_error'] = traceback.format_exc()[-2500:]
    finally:
        srv.stop(); time.sleep(15)
    return rec

## Stage A: baseline serving, no adapters

In [ ]:
# Baseline measured in run 1 (same notebook, same wheelhouse, same day): reused to save ~10 min of L4x4. Source: results of kernel version 1.
results['A_baseline_serving'] = {'reused_from': 'g1 run 1', 'boot_s': 534,
    'kv_lines': ['Available KV cache memory: 10.61 GiB', 'GPU KV cache size: 46,352 tokens', 'Maximum concurrency for 32,768 tokens per request: 8.74x'],
    'decode': {'BASE': {'decode_tok_s_median': 37.71, 'ttft_s_median': 0.08}}, 'concurrency4': {'BASE': {'aggregate_tok_s': 131.9, 'wall_s': 6.07}},
    'cmd': '--max-model-len 32768 --gpu-memory-utilization 0.9 --tensor-parallel-size 4 --reasoning-config {...} --no-scheduler-reserve-full-isl'}
save_results(); print('baseline reused:', results['A_baseline_serving']['decode'], results['A_baseline_serving']['kv_lines'])

## Stage B: training spike

In [ ]:
import g1_util as U
t0 = time.time()
STAGED = Path('/tmp/ckpt_bf16')
try:   # /kaggle/input is NFS (~90 MB/s/stream, ~320 MB/s with 4 parallel streams; transformers' mmap reads crawled at ~6 MB/s): stage to local disk first
    results['B_stage'] = U.stage_checkpoint(MODEL_BF16, STAGED, workers=12)
    print('staged:', results['B_stage'])
except Exception as e:
    results['B_stage'] = {'error': f'{type(e).__name__}: {e}'}; print('STAGING FAILED', e)
save_results()
if 'error' not in results['B_stage']:
    env = dict(os.environ, G1_MODEL=str(STAGED), G1_DATA=str(G1_DATA), G1_OUT=str(G1), G1_MAXLEN='16384', G1_MICRO='12', G1_ACC='4',
               G1_MAXMEM=json.dumps({'0': '16GiB', '1': '16GiB', '2': '16GiB', '3': '16GiB'}), G1_TRAIN_SECONDS='1200',
               G1_PROBES='16384,24576,32768', TOKENIZERS_PARALLELISM='false')
    # milestones: weights loaded within 10 min (local disk), first training step within 20 min of start; whole stage <= 55 min
    results['B_run'] = U.run_supervised([sys.executable, str(CODE / 'train_g1.py')], env, G1 / 'train.log',
                                        [('[g1] loaded in', 600), ('[g1] micro 0', 1200)], max_seconds=55 * 60)
    print(results['B_run'])
tm = G1 / 'train_metrics.json'
results['B_train'] = json.loads(tm.read_text()) if tm.exists() else {'error': 'no train_metrics.json'}
results['B_seconds'] = round(time.time() - t0); save_results()
shutil.rmtree(STAGED, ignore_errors=True)
print(subprocess.run(f'grep -F "[g1]" {G1}/train.log | tail -40', shell=True, capture_output=True, text=True).stdout)
b = results['B_train']; print({k: b.get(k) for k in ('stage', 'tokens_per_second', 'peak_gib_per_gpu', 'load_seconds', 'trainable_params_m', 'error')}); print('probes:', b.get('probes'))

## Stage C: adapter serving

In [ ]:
t0 = time.time()
src = G1 / 'adapter_full'
if not (src / 'adapter_model.safetensors').exists():
    results['C_adapter_serving'] = {'skipped': 'no trained adapter (stage B failed)'}
else:
    import make_adapters
    try:
        results['C_adapters'] = make_adapters.main(src, G1 / 'adapters', DATA_DIR / 'sample_submission' / 'adapters' / 'main_lora')
    except Exception:
        results['C_adapters'] = {'error': traceback.format_exc()[-2000:]}
    save_results()
    results['C_adapter_serving'] = None
    for names in (['g1_full', 'g1_attn', 'g1_canary'], ['g1_attn', 'g1_canary']):   # fall back to attention-only if MLP LoRA is unsupported by this vLLM
        rec = serve_and_probe('lora_' + '_'.join(n[3:] for n in names), names)
        results['C_adapter_serving'] = rec
        save_results()
        if 'start_error' not in rec: break
results['C_seconds'] = round(time.time() - t0); save_results()
c = results['C_adapter_serving']; print(json.dumps({k: c.get(k) for k in ('adapters', 'boot_s', 'kv_lines', 'lora_log_lines', 'start_error', 'probe_error', 'skipped')}, indent=1)[:3500])
for m, d in (c.get('decode') or {}).items(): print('decode', m, d.get('decode_tok_s_median'), 'tok/s; ttft', d.get('ttft_s_median'))
print('long prompt:', c.get('long_prompt')); print('adapter activity:', {a: (v['max_logprob_diff'], v['any_text_differs']) for a, v in (c.get('adapter_activity') or {}).items()})

## Summary and gates

In [ ]:
def g(d, *ks):
    for k in ks:
        d = (d or {}).get(k) if isinstance(d, dict) else None
    return d
A, B, C = results.get('A_baseline_serving') or {}, results.get('B_train') or {}, results.get('C_adapter_serving') or {}
def first_decode(r, adapter=False):
    ds = r.get('decode') or {}
    for m, d in ds.items():
        if adapter == (m.startswith('g1_')): return d.get('decode_tok_s_median')
base_tps = first_decode(A); ad_tps = None
for m, d in (C.get('decode') or {}).items():
    if m == 'g1_attn' or (ad_tps is None and m.startswith('g1_') and m != 'g1_canary'): ad_tps = d.get('decode_tok_s_median')
gates = {
  'train tokens/s >= 60': (B.get('tokens_per_second') or 0) >= 60,
  'training fits 16k tokens (no OOM events)': not B.get('oom_events'),
  'adapter active in serving (canary logprob diff > 0.05)': (g(C, 'adapter_activity', 'g1_canary', 'max_logprob_diff') or 0) > 0.05,
  'adapter decode >= 85% of no-adapter baseline': bool(base_tps and ad_tps and ad_tps / base_tps >= 0.85),
  'long prompt completes with adapter': bool(g(C, 'long_prompt', 'g1_attn', 'ok') or g(C, 'long_prompt', 'g1_full', 'ok')),
}
print('=== G1 SUMMARY ===')
print('baseline decode tok/s:', base_tps, '| adapter(attn) decode tok/s:', ad_tps, '| ratio:', round(ad_tps / base_tps, 3) if base_tps and ad_tps else None)
print('train tok/s:', B.get('tokens_per_second'), '| peak GiB/GPU:', B.get('peak_gib_per_gpu'), '| probes:', [(p.get('len'), p.get('ok')) for p in (B.get('probes') or [])])
print('KV lines baseline:', A.get('kv_lines')); print('KV lines with adapters:', C.get('kv_lines'))
for k, v in gates.items(): print(('PASS ' if v else 'FAIL ') + k)
results['gates'] = gates; results['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); save_results()
for d in WORK.glob('sub_*'): shutil.rmtree(d, ignore_errors=True)   # keep the output small
_wd.cancel()